# AGAR-RL: Autonomous Multi-Agent Deep Reinforcement Learning Pipeline

Ce notebook permet d'exécuter l'entraînement distribué par Deep Reinforcement Learning (PPO & Self-Play) directement depuis **VS Code** (via l'extension Google Colab ou votre kernel local) ou sur **Google Colab web**.

## 1. Détection de l'Environnement et Récupération du Code

In [ ]:
import os, sys

# 1. Mise à jour ou clonage du code source
if os.path.exists("agario/.git"):
    print("🔄 Mise à jour du dépôt agario existant...")
    %cd agario
    !git pull origin main
elif os.path.exists(".git") and os.path.exists("src/training/train_colab.py"):
    print("✅ Exécution locale dans le workspace agario.")
else:
    print("🌐 Environnement distant Colab détecté. Clonage du repo...")
    res = os.system("git clone https://github.com/Albin0903/agario.git")
    
    # Si le repo est privé, demande de token ou utilisation du secret Colab
    if res != 0 or not os.path.exists("agario"):
        print("\n⚠️ Le repo est PRIVÉ sur GitHub.")
        print("👉 Option A : Passez le repo en Public sur GitHub (Settings > Change visibility > Public).")
        print("👉 Option B : Entrez un GitHub Personal Access Token (classic avec permission 'repo') :\n")
        token = ""
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            pass
        
        if not token:
            import getpass
            token = getpass.getpass("Token GitHub (ou appuyez sur Entrée si vous l'avez passé en public) : ").strip()
            
        if token:
            !git clone https://{token}@github.com/Albin0903/agario.git
        else:
            !git clone https://github.com/Albin0903/agario.git
            
    if os.path.exists("agario"):
        %cd agario
    else:
        raise RuntimeError("Impossible d'accéder au dossier agario. Vérifiez la visibilité de votre repo ou votre token.")

# 2. Configuration du PYTHONPATH et installation des dépendances
os.environ["PYTHONPATH"] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip install -q -r requirements.txt tensorboard

# 3. Vérification GPU CUDA
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU actif : {torch.cuda.get_device_name(0)}")
else:
    print("Exécution sur CPU.")

## 2. Validation des Tests Unitaires (20 Tests)

In [ ]:
# Exécution avec python -m pytest pour garantir la résolution de src
!python -m pytest -v

## 3. Monitoring TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard

## 4. Démarrer l'Entraînement PPO & Self-Play

In [ ]:
# Lance 16 environnements en parallèle avec 10 bots par arène et mise à jour du pool d'adversaires
!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 1000000 \
    --batch-size 128 \
    --n-steps 2048 \
    --pool-interval 200000 \
    --device auto

## 5. Exporter la Politique vers ONNX (< 0.02 ms de latence CPU)

In [ ]:
!python src/inference/export_onnx.py \
    --model checkpoints/ppo/ppo_final.zip \
    --output models/model.onnx

## 6. Téléchargement du Modèle (si exécuté sur VM Colab distante)

In [ ]:
try:
    from google.colab import files
    files.download('models/model.onnx')
    files.download('checkpoints/ppo/ppo_final.zip')
    print("Téléchargement Colab initié.")
except ImportError:
    print("Fichiers sauvegardés localement dans models/ et checkpoints/.")